# Artigo 3 — execução canônica no Colab

Runtime: **GPU T4** (Ambiente de execução → Alterar o tipo de ambiente).
Siga as células em ordem. Nenhum parâmetro experimental é definido aqui: tudo vem de `src/config.py`.

Antes de começar: o `config.py` do SHA abaixo não pode ter nenhum `PENDENTE` (inclusive `dataset.doi`).
Os resultados vão direto para o Drive; se o runtime cair, reabra o notebook e rode tudo de novo — a geração retoma de onde parou.

In [ ]:
SHA = '<SHA-DO-COMMIT-CONGELADO>'          # commit da main com o DOI preenchido
PDFS = '/content/drive/MyDrive/IA2/art3'    # os 10 PDFs das aulas 14–23
SAIDA = '/content/drive/MyDrive/IA2/art3-resultados'

## 1. Código, ambiente e dados

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
assert SHA != '<SHA-DO-COMMIT-CONGELADO>', 'preencha o SHA'
if not os.path.isdir('/content/IA2-2026.2'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/HeitorM50/IA2-2026.2.git', '/content/IA2-2026.2'], check=True)
%cd /content/IA2-2026.2
!git checkout -q {SHA} && git log --oneline -1
%cd artigo-3
os.environ['IA2_CORPUS_DIR'] = PDFS
os.makedirs(SAIDA, exist_ok=True)

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Verificações (devem passar sem erro e sem teste pulado)

In [ ]:
!python -m pytest -q
!python -m src.corpus
!git diff --exit-code data/manifesto-corpus.json && echo 'corpus idêntico ao do commit'
!git status --porcelain && echo '(acima: nada = Git limpo)'

## 3. Recuperação no dev e seleção de k (o test não é tocado aqui)

In [ ]:
DEV = f'{SAIDA}/retrieval/dev'
!python -m src.avaliacao_recuperacao avaliar --split dev --saida {DEV}
!python -m src.avaliacao_recuperacao selecionar-k --avaliacao-dir {DEV}
SEL = f'{DEV}/top-k-selection.json'
!python -c "import json; print('k =', json.load(open('{SEL}'))['top_k_selecionado'])"

## 4. Ensaio de geração no dev

Conferir à mão: o modelo responde à pergunta, cita `[n]` e se abstém na não respondível.
Se algo estiver errado, **pare aqui** — corrigir exige commit novo e recomeçar com o SHA novo.

In [ ]:
ENSAIO = f'{SAIDA}/raw/ensaio'
!python -m src.run --split dev --limite 3 --configs slide --saida {ENSAIO} --selecao-top-k {SEL} --resume
import json, glob
for f in sorted(glob.glob(f'{ENSAIO}/*/*.json')):
    r = json.load(open(f))
    if 'resposta' in r:
        print(f"{r['pergunta_id']} | abstencao={r['abstencao']} citacoes={r['citacoes']}\nP: {r['pergunta']}\nR: {r['resposta']}\n")

## 5. Grade canônica: 7 configurações × 45 perguntas = 315 pares

In [ ]:
!python -m src.avaliacao_recuperacao avaliar --split test --saida {SAIDA}/retrieval/test --selecao-top-k {SEL}
!python -m src.run --split test --saida {SAIDA}/raw/test --selecao-top-k {SEL} --resume

## 6. Métricas automáticas e amostra para a avaliação humana

In [ ]:
!python -m src.analise_geracao automaticas --split test --resultados {SAIDA}/raw | tee {SAIDA}/geracao-automaticas.json
!python -m src.analise_geracao amostra --split test --n 10 --resultados {SAIDA}/raw --destino {SAIDA}/avaliacao
!ls -R {SAIDA} | head -40

Pronto. Tudo está em `SAIDA` no Drive: `run-manifest.json`, `environment.json`, `pip-freeze.txt`, `run.log`,
`summary.csv` da recuperação, `top-k-selection.json`, os JSON brutos e a planilha cega (`avaliacao/planilha-cega.csv`).
**Não abra `avaliacao/chave.csv`** antes de terminar a avaliação.